In [10]:
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Conv2DTranspose, Dropout
from keras.utils import normalize
#from sklearn.preprocessing import normalize
import os
import glob
import cv2
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from matplotlib import pyplot as plt

In [11]:
# U-Net model definition
def multi_unet_model(n_classes=4, IMG_HEIGHT=512, IMG_WIDTH=512, IMG_CHANNELS=3):
    inputs = Input((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
    s = inputs

    # Contraction path
    c1 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(s)
    c1 = Dropout(0.1)(c1)
    c1 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p1)
    c2 = Dropout(0.1)(c2)
    c2 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p2)
    c3 = Dropout(0.2)(c3)
    c3 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c3)
    p3 = MaxPooling2D((2, 2))(c3)

    c4 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p3)
    c4 = Dropout(0.2)(c4)
    c4 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c4)
    p4 = MaxPooling2D(pool_size=(2, 2))(c4)

    c5 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p4)
    c5 = Dropout(0.3)(c5)
    c5 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c5)

    # Expansive path
    u6 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = concatenate([u6, c4])
    c6 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u6)
    c6 = Dropout(0.2)(c6)
    c6 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c6)

    u7 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = concatenate([u7, c3])
    c7 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u7)
    c7 = Dropout(0.2)(c7)
    c7 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c7)

    u8 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = concatenate([u8, c2])
    c8 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u8)
    c8 = Dropout(0.1)(c8)
    c8 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c8)

    u9 = Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = concatenate([u9, c1], axis=3)
    c9 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u9)
    c9 = Dropout(0.1)(c9)
    c9 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c9)

    outputs = Conv2D(n_classes, (1, 1), activation='softmax')(c9)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model


In [12]:
# Data loading
SIZE_X = 512
SIZE_Y = 512
n_classes = 4


In [13]:
# Load colored images
image_dir = "images_data/split_colored_raw_dataset"
mask_dir = "images_data/split_segmented_raw_dataset"

In [34]:
train_images = []
for img_path in glob.glob(os.path.join(image_dir, "*.tif")):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Read in RGB
    #img = cv2.resize(img, (SIZE_X, SIZE_Y))  # Resize if necessary
    train_images.append(img)
train_images = np.array(train_images)

In [35]:
print("Train Images size: ", np.size(train_images))
print("Train Images shape: ", np.shape(train_images))
print("Train Images dtype: ", train_images.dtype)


Train Images size:  257163264
Train Images shape:  (981, 512, 512)
Train Images dtype:  uint8


In [16]:
# Load segmented images
train_masks = []
for mask_path in glob.glob(os.path.join(mask_dir, "*.tif")):
    mask = cv2.imread(mask_path, 0)
    #train_masks.append(cv2.resize(mask, (SIZE_X, SIZE_Y), interpolation=cv2.INTER_NEAREST))
    train_masks.append(mask)
train_masks = np.array(train_masks)
#normalized_masks = train_masks // 85

In [17]:
print("Train Masks shape: ",np.shape(train_masks))
print("Train Masks dtype: ", train_masks.dtype)
#print(np.shape(normalized_masks))
print("Train Masks classes: ", np.unique(train_masks))
#np.unique(normalized_masks)

Train Masks shape:  (981, 512, 512)
Train Masks dtype:  uint8
Train Masks classes:  [0 1 2 3]


In [13]:
print(np.shape(train_images))

(981, 512, 512, 3)


In [38]:
# Normalize and expand dimensions
train_images = np.expand_dims(train_images, axis = 3)
train_images = normalize(train_images, axis=1)   #increase dimension by 1 (981x512x512) to (981x512x512x1) to be suitable for U-net model

train_masks_input = np.expand_dims(train_masks, axis=3)

In [40]:
print("After Normalization: \n")
print("Train Images dtype: ", train_images.dtype)
print("Train Masks_input shape: ", np.shape(train_masks_input))
print("Train Images shape: ", np.shape(train_images))

After Normalization: 

Train Images dtype:  float64
Train Masks_input shape:  (981, 512, 512, 1)
Train Images shape:  (981, 512, 512, 1)


In [20]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(train_images, train_masks_input, test_size=0.10, random_state=42)



In [22]:
print(" X_train shape: ",np.shape(X_train))
print(" X_test shape: ",np.shape(X_test))
print(" y_train shape: ",np.shape(y_train))
print("  y_test shape: ",np.shape(y_test))
print(" y_train classes: ",np.unique(y_train))
print("  y_test classes: ",np.unique(y_test))

 X_train shape:  (882, 512, 512, 3)
 X_test shape:  (99, 512, 512, 3)
 y_train shape:  (882, 512, 512, 1)
  y_test shape:  (99, 512, 512, 1)
 y_train classes:  [0 1 2 3]
  y_test classes:  [0 1 2 3]


In [23]:
# One-hot encoding of masks

y_train_cat = to_categorical(y_train, num_classes=n_classes)
y_test_cat = to_categorical(y_test, num_classes=n_classes)


In [24]:
print("After One hot incoding: \n")

print("y_train_cat shaape: ", np.shape(y_train_cat))
print("y_train_cat dtype: ", y_train_cat.dtype)
print("y_test_cat shape", np.shape(y_test_cat))

After One hot incoding: 

y_train_cat shaape:  (882, 512, 512, 4)
y_train_cat dtype:  float64
y_test_cat shape (99, 512, 512, 4)


In [53]:
#train_masks_reshaped = train_masks.reshape(-1,1)
#print(np.shape(train_masks_reshaped))
#print(np.unique(train_masks_reshaped))
print(np.shape(train_masks))

(981, 512, 512)


In [ ]:
# Label encoding
labelencoder = LabelEncoder()

train_masks_reshaped = train_masks.reshape(-1, 1)
train_masks_reshaped_encoded = labelencoder.fit_transform(train_masks_reshaped)

C:\Users\Vinit\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [54]:
y = [2,3,4,5,6,7,7,7,7]
print(np.unique(y))
print(len(y))
print(np.shape(train_masks_reshaped))

[2 3 4 5 6 7]
9
(257163264, 1)


In [27]:

print("Encoded into single vector to assign class_weights: (will be used only when class_weights are provided in history.fit) \n")
print("train_masks_reshaped_encoded shape", np.shape(train_masks_reshaped_encoded))
print("train_masks_reshaped_encoded classes", np.unique(train_masks_reshaped_encoded))

Encoded into single vector to assign class_weights: (will be used only when class_weights are provided in history.fit) 

train_masks_reshaped_encoded shape (257163264,)
train_masks_reshaped_encoded classes [0 1 2 3]


In [56]:
sample_weights = np.zeros(len(train_masks_reshaped_encoded))
sample_weights[train_masks_reshaped_encoded == 0] = 3.686716245843107
sample_weights[train_masks_reshaped_encoded == 1] = 1.3096818001509811
sample_weights[train_masks_reshaped_encoded == 2] = 1.069314643545103
sample_weights[train_masks_reshaped_encoded == 3] = 33.29645823714349

In [57]:
# Reshape sample_weights to match the shape of the masks (num_samples, height, width)
sample_weights = sample_weights.reshape(train_masks.shape)  # Shape: (num_samples, height, width)
print("sample_weight shape: ", np.shape(sample_weights))
# Split sample_weights into training and validation sets
sample_weights_train, sample_weights_test = train_test_split(sample_weights, test_size=0.10, random_state=42)
print("sample_weight_train shape: ", np.shape(sample_weights_train))

sample_weight shape:  (981, 512, 512)
sample_weight_train shape:  (882, 512, 512)


In [58]:
print(sample_weights_train[0:10])

[[[ 1.3096818   1.3096818   1.3096818  ...  1.3096818   1.3096818
    1.3096818 ]
  [ 1.3096818   1.3096818   1.3096818  ...  1.3096818   1.3096818
    1.3096818 ]
  [ 1.3096818   1.3096818   1.3096818  ...  1.3096818   1.3096818
    1.3096818 ]
  ...
  [ 1.06931464  1.06931464  1.06931464 ...  1.06931464  1.06931464
    1.06931464]
  [ 1.06931464  1.06931464  1.06931464 ...  1.06931464  1.06931464
    1.06931464]
  [ 1.06931464  1.06931464  1.06931464 ...  1.06931464  1.06931464
    1.06931464]]

 [[ 1.3096818   1.3096818   1.3096818  ...  1.3096818   1.3096818
    1.3096818 ]
  [ 1.3096818   1.3096818   1.3096818  ...  1.3096818   1.3096818
    1.3096818 ]
  [ 1.3096818   1.3096818   1.3096818  ...  1.3096818   1.3096818
    1.3096818 ]
  ...
  [ 1.06931464  1.06931464  1.06931464 ...  1.06931464  1.06931464
    1.06931464]
  [ 1.06931464  1.06931464  1.06931464 ...  1.06931464  1.06931464
    1.06931464]
  [ 1.06931464  1.06931464  1.06931464 ...  1.06931464  1.06931464
    1.069314

In [52]:

import numpy as np

# Assuming train_masks is a NumPy array
unique_values, counts = np.unique(train_masks_reshaped_encoded, return_counts=True)

# Print the counts for each unique value
for value, count in zip(unique_values, counts):
    print(f"Number of {value}'s in train_masks: {count}")

Number of 0's in train_masks: 34877008
Number of 1's in train_masks: 98177765
Number of 2's in train_masks: 120246770
Number of 3's in train_masks: 3861721


In [28]:
from sklearn.utils import class_weight
class_weights_a = class_weight.compute_class_weight(class_weight= 'balanced', 
                                                  classes= np.unique(train_masks_reshaped_encoded),
                                                  y= train_masks_reshaped_encoded)


In [29]:
class_weights = {i: weight for i, weight in enumerate(class_weights_a)}

In [30]:
print("class weights are... ", class_weights)    # labels might be :  0 = 0 deg Fiber, 1 = 90 deg Fiber, 2 = Resin, 3 = Other,

class weights are...  {0: 1.8433581229215534, 1: 0.6548409000754906, 2: 0.5346573217725515, 3: 16.648229118571745}


In [55]:
print(np.shape(train_masks))
print(np.shape(X_train))

(981, 512, 512)
(882, 512, 512, 3)


In [32]:
IMG_HEIGHT = X_train.shape[1]
IMG_WIDTH = X_train.shape[2]
IMG_CHANNELS = X_train.shape[3]

In [41]:
print(IMG_HEIGHT , IMG_WIDTH, IMG_CHANNELS)

512 512 3


In [33]:
# Model creation
model = multi_unet_model(n_classes=n_classes, IMG_HEIGHT=IMG_HEIGHT, IMG_WIDTH=IMG_WIDTH, IMG_CHANNELS= IMG_CHANNELS)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])   #since we are using multiclass we need to use categorial_crossentr.., softmax 
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 512, 512,  │        448 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512, 512,  │          0 │ conv2d[0][0]      │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 512, 512,  │      2,320 │ dropout[0][0]     │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 256, 256,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 256, 256,  │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256, 256,  │          0 │ conv2d_2[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 256, 256,  │      9,248 │ dropout_1[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 128, 128,  │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 128, 128,  │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128, 128,  │          0 │ conv2d_4[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 128, 128,  │     36,928 │ dropout_2[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 64, 64,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64, 64,    │          0 │ conv2d_6[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 64, 64,    │    147,584 │ dropout_3[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 32, 32,    │          0 │ conv2d_7[0][0]  

 Total params: 1,941,156 (7.40 MB)

 Trainable params: 1,941,156 (7.40 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:

# Training
history = model.fit(X_train, y_train_cat, batch_size=16, verbose=2, epochs=50, validation_data=(X_test, y_test_cat), shuffle = False)

KeyboardInterrupt: 

In [73]:




# Save model
model.save('segmentation_model.hdf5')